# Predicting LYVE1 and CCL19 expression in unseen endometriosis patients (ML part)

Runs in Google Colab. Reads the files exported by the R pipeline (`scripts/01_r_pipeline`,
`scripts/02_external_prep`) from the Google Drive folder `endo_ml`, and writes all results to
`endo_ml/ml_outputs/` (tables, predictions, figures).

**Inputs in the Drive folder `endo_ml`:**
- `macrophages_LYVE1.zip` - lognorm.mtx, genes.txt, barcodes.txt, cell_metadata.csv (4,762 macrophages)
- `myeloid_clusters.csv` - myeloid cluster of every macrophage (used for the cluster-mean baseline)
- `mural_CCL19/` - folder with lognorm.mtx, genes.txt, barcodes.txt, cell_metadata.csv (2,537 mural cells)
- `external_macrophages.zip` - same four files for the external cohort (1,663 macrophage-like cells)

**Design:** one model per target gene; leave-one-patient-out evaluation; the target gene is removed from the
inputs; the 2,000 most variable genes are chosen on training cells only. A patient is tested only if it has
at least 50 cells and 10 expressing cells. Models: Elastic Net, Random Forest, XGBoost.
Predictions reflect association, not causal mechanism.

In [ ]:
# Settings
RUN_RF = True        # Random Forest is the slowest model; set False for a faster run
RUN_TUNING = True    # nested tuning of Elastic Net (a few minutes)
SEED = 42

In [ ]:
import os, glob, zipfile, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from itertools import product
from scipy.io import mmread
from scipy.stats import spearmanr
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from xgboost import XGBRegressor

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/endo_ml"
    WORK = "/content/data"
except ImportError:                                   # running outside Colab
    ROOT = os.environ.get("ENDO_ROOT", ".")
    WORK = os.environ.get("ENDO_WORK", "./work")
OUT = os.path.join(ROOT, "ml_outputs")
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)
assert os.path.isdir(ROOT), "Drive folder endo_ml not found - run again and press Allow"

## 1. Load the data

In [ ]:
def unpack_zip(pattern, dest):
    hits = glob.glob(os.path.join(ROOT, pattern))
    assert len(hits) == 1, f"{pattern}: found {hits}"
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(hits[0]) as z:
        z.extractall(dest)

def read_dataset(d):
    X = mmread(os.path.join(d, "lognorm.mtx")).tocsr().T.tocsr()      # cells x genes (log-normalized)
    genes = open(os.path.join(d, "genes.txt")).read().splitlines()
    bc = open(os.path.join(d, "barcodes.txt")).read().splitlines()
    meta = pd.read_csv(os.path.join(d, "cell_metadata.csv"))
    assert X.shape == (len(bc), len(genes)) and (meta["barcode"].values == np.array(bc)).all()
    return X, genes, meta

unpack_zip("macrophages_LYVE1*.zip", os.path.join(WORK, "mac"))
unpack_zip("external_macrophages*.zip", os.path.join(WORK, "ext"))
X_mac, genes, meta_mac = read_dataset(os.path.join(WORK, "mac"))
X_mur, genes_m, meta_mur = read_dataset(os.path.join(ROOT, "mural_CCL19"))
X_ext, genes_e, meta_ext = read_dataset(os.path.join(WORK, "ext"))
assert genes == genes_m == genes_e, "gene order differs between datasets"
print("macrophages:", X_mac.shape, "| mural:", X_mur.shape, "| external:", X_ext.shape)
# expected (4762, 27722) (2537, 27722) (1663, 27722)

## 2. Helper functions

In [ ]:
def top_var_genes(X, n=2000):                          # computed on TRAIN cells only
    m = np.asarray(X.mean(axis=0)).ravel()
    m2 = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    return np.argsort(m2 - m ** 2)[::-1][:n]

def metrics(yt, yp):
    r = dict(spearman=spearmanr(yt, yp)[0], r2=r2_score(yt, yp),
             rmse=mean_squared_error(yt, yp) ** 0.5, mae=mean_absolute_error(yt, yp),
             bias=float(np.mean(yp) - np.mean(yt)))
    pos = yt > 0
    r["auc"] = roc_auc_score(pos, yp) if 0 < pos.sum() < len(yt) else np.nan   # expressing vs zero
    r["sp_pos"] = spearmanr(yt[pos], yp[pos])[0] if pos.sum() >= 5 else np.nan # ranking among expressing cells
    return r

def usable_patients(y, pid, min_cells=50, min_pos=10):
    t = pd.DataFrame({"p": pid, "pos": y > 0}).groupby("p")["pos"].agg(["size", "sum"])
    return t.index[(t["size"] >= min_cells) & (t["sum"] >= min_pos)].tolist()

MODELS = {
    "elastic_net": lambda: ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=2000),
    "random_forest": lambda: RandomForestRegressor(n_estimators=150, max_features=0.1, min_samples_leaf=5,
                                                   n_jobs=-1, random_state=SEED),
    "xgboost": lambda: XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                                    colsample_bytree=0.5, tree_method="hist", random_state=SEED),
}
if not RUN_RF:
    MODELS.pop("random_forest")
FAST_MODELS = {k: MODELS[k] for k in ["elastic_net", "xgboost"]}

def prep_fold(X, keep, tr, te):
    Xtr_all = X[tr][:, keep]
    sel = top_var_genes(Xtr_all)
    Xtr = Xtr_all[:, sel].toarray()
    Xte = X[te][:, keep][:, sel].toarray()
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    return (Xtr - mu) / sd, (Xte - mu) / sd, sel

def lopo(X, y, pid, base_labels, keep, use, makers, tag):
    """Leave-one-patient-out. base_labels: groups for the baseline (mean of the training cells of each group)."""
    pred = {n: np.full(len(y), np.nan) for n in list(makers) + ["baseline"]}
    rows = []
    for p in use:
        te = np.where(pid == p)[0]; tr = np.where(pid != p)[0]
        Xtr, Xte, _ = prep_fold(X, keep, tr, te)
        gm = {g: y[tr][base_labels[tr] == g].mean() for g in np.unique(base_labels)}
        pred["baseline"][te] = [gm[g] for g in base_labels[te]]
        for n, make in makers.items():
            pred[n][te] = make().fit(Xtr, y[tr]).predict(Xte)
        for n in pred:
            rows.append(dict(analysis=tag, patient=p, model=n, n=len(te),
                             n_pos=int((y[te] > 0).sum()), **metrics(y[te], pred[n][te])))
        print("done", tag, p)
    return pd.DataFrame(rows), pred

def summarize(res):
    return res.groupby("model")[["spearman", "r2", "rmse", "mae", "auc", "sp_pos"]].mean().round(3)

def tech_only(meta, y, pid, use):
    """Baseline that sees only technical metrics (UMI count, genes detected, mitochondrial fraction)."""
    T = np.column_stack([np.log1p(meta["nCount_RNA"].values), np.log1p(meta["nFeature_RNA"].values),
                         meta["percent.mt"].values])
    pred = np.full(len(y), np.nan)
    for p in use:
        te = np.where(pid == p)[0]; tr = np.where(pid != p)[0]
        mu, sd = T[tr].mean(0), T[tr].std(0) + 1e-8
        m = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                         tree_method="hist", random_state=SEED)
        pred[te] = m.fit((T[tr] - mu) / sd, y[tr]).predict((T[te] - mu) / sd)
    return pred

def tune_en(X, y, pid, keep, use, tag):
    """Elastic Net with nested tuning: the setting is chosen inside the training patients only."""
    grid = [dict(alpha=a, l1_ratio=l) for a, l in product([0.03, 0.05, 0.1, 0.2], [0.5, 0.8])]
    rows, chosen = [], []
    for p in use:
        te = np.where(pid == p)[0]; tr = np.where(pid != p)[0]
        Xtr, Xte, _ = prep_fold(X, keep, tr, te)
        ytr = y[tr]
        inner = list(GroupKFold(n_splits=3).split(Xtr, ytr, groups=pid[tr]))
        best, best_s = None, -9
        for prm in grid:
            s = []
            for itr, iva in inner:
                pr = ElasticNet(max_iter=2000, **prm).fit(Xtr[itr], ytr[itr]).predict(Xtr[iva])
                r = spearmanr(ytr[iva], pr)[0]
                s.append(0 if np.isnan(r) else r)
            if np.mean(s) > best_s:
                best, best_s = prm, np.mean(s)
        pr = ElasticNet(max_iter=2000, **best).fit(Xtr, ytr).predict(Xte)
        rows.append(dict(analysis=tag, patient=p, model="elastic_net_tuned", n=len(te),
                         n_pos=int((y[te] > 0).sum()), **metrics(y[te], pr)))
        chosen.append(dict(patient=p, **best))
        print("done tuning", tag, p)
    return pd.DataFrame(rows), pd.DataFrame(chosen)

def stable_genes(X, y, pid, keep, gene_names, use):
    """Genes with a non-zero Elastic Net coefficient in every leave-one-patient-out fold."""
    rows = []
    for p in use:
        tr = np.where(pid != p)[0]
        Xtr_all = X[tr][:, keep]
        sel = top_var_genes(Xtr_all)
        Xtr = Xtr_all[:, sel].toarray()
        Xtr = (Xtr - Xtr.mean(0)) / (Xtr.std(0) + 1e-8)
        en = ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=2000).fit(Xtr, y[tr])
        rows += [(p, gene_names[g], c) for g, c in zip(keep[sel], en.coef_)]
    cf = pd.DataFrame(rows, columns=["fold", "gene", "coef"])
    agg = (cf.assign(nz=cf["coef"] != 0, pos=cf["coef"] > 0).groupby("gene")
             .agg(folds_selected=("nz", "sum"), mean_coef=("coef", "mean"), n_pos=("pos", "sum")))
    st = agg[agg["folds_selected"] == len(use)].copy()
    st["same_sign_all_folds"] = (st["n_pos"] == 0) | (st["n_pos"] == len(use))
    return st.drop(columns="n_pos").sort_values("mean_coef", ascending=False), len(agg)

def save_preds(meta, y, pred, target, path):
    out = meta[["barcode", "patient_id", "tissue", "subtype"]].copy() if "subtype" in meta else \
          meta[["barcode", "patient_id", "tissue"]].copy()
    out[target] = y
    for k, v in pred.items():
        out["pred_" + k] = v
    out.to_csv(path, index=False)

## 3. LYVE1 in macrophages: leave-one-patient-out (internal evaluation)

In [ ]:
j_lv = genes.index("LYVE1")
y_lv = X_mac[:, j_lv].toarray().ravel()
keep_lv = np.array([i for i in range(len(genes)) if i != j_lv])        # target gene excluded
pid_lv = meta_mac["patient_id"].values
sub_lv = meta_mac["subtype"].values.astype(str)
use_lv = usable_patients(y_lv, pid_lv)
print(len(use_lv), "usable patients:", use_lv)                          # expected 10

res_lv, pred_lv = lopo(X_mac, y_lv, pid_lv, sub_lv, keep_lv, use_lv, MODELS, "LYVE1_macrophages")
print(summarize(res_lv))
res_lv.to_csv(os.path.join(OUT, "lyve1_internal_per_patient.csv"), index=False)
save_preds(meta_mac, y_lv, pred_lv, "LYVE1", os.path.join(OUT, "preds_macrophages_LYVE1.csv"))

## 4. CCL19 in mural cells: leave-one-patient-out (internal evaluation)

In [ ]:
j_cc = genes.index("CCL19")
y_cc = X_mur[:, j_cc].toarray().ravel()
keep_cc = np.array([i for i in range(len(genes)) if i != j_cc])
pid_cc = meta_mur["patient_id"].values
sub_cc = meta_mur["subtype"].values.astype(str)
use_cc = usable_patients(y_cc, pid_cc)
print(len(use_cc), "usable patients:", use_cc)                          # expected 7

res_cc, pred_cc = lopo(X_mur, y_cc, pid_cc, sub_cc, keep_cc, use_cc, MODELS, "CCL19_mural")
print(summarize(res_cc))
res_cc.to_csv(os.path.join(OUT, "ccl19_internal_per_patient.csv"), index=False)
save_preds(meta_mur, y_cc, pred_cc, "CCL19", os.path.join(OUT, "preds_mural_CCL19.csv"))

## 5. Robustness checks

### 5a. LYVE1 without the LYVE1-defined cluster (Other macrophages only, cluster-mean baseline)

In [ ]:
cl_df = pd.read_csv(os.path.join(ROOT, "myeloid_clusters.csv"))
cl_all = meta_mac["barcode"].map(dict(zip(cl_df["barcode"], cl_df["myeloid_cluster"].astype(str)))).values
om = np.where(sub_lv == "Other macrophage")[0]
cl_om = cl_all[om].astype(str)
assert not pd.isna(cl_all[om]).any(), "some macrophages have no cluster label"
y_om, pid_om = y_lv[om], pid_lv[om]
use_om = usable_patients(y_om, pid_om)
print(len(use_om), "usable patients (Other macrophage only):", use_om)   # expected 9

res_om, pred_om = lopo(X_mac[om], y_om, pid_om, cl_om, keep_lv, use_om, FAST_MODELS, "LYVE1_other_macrophage")
print(summarize(res_om))                                                  # 'baseline' = cluster mean
res_om.to_csv(os.path.join(OUT, "lyve1_other_macrophage_per_patient.csv"), index=False)

print("\nSpearman inside each cluster (test patients pooled):")
for c in np.unique(cl_om):
    m = (cl_om == c) & np.isin(pid_om, use_om)
    print("cluster", c, "| cells:", int(m.sum()),
          "| EN: %.3f" % spearmanr(y_om[m], pred_om["elastic_net"][m])[0],
          "| XGB: %.3f" % spearmanr(y_om[m], pred_om["xgboost"][m])[0])

### 5b. Technical metrics only (UMI count, genes detected, mitochondrial fraction)

In [ ]:
tech_lv = tech_only(meta_mac, y_lv, pid_lv, use_lv)
m = np.isin(pid_lv, use_lv)
print("LYVE1, all macrophages: technical-only pooled Spearman %.3f | XGBoost with genes %.3f" %
      (spearmanr(y_lv[m], tech_lv[m])[0], spearmanr(y_lv[m], pred_lv["xgboost"][m])[0]))

tech_cc = tech_only(meta_mur, y_cc, pid_cc, use_cc)
prv = sub_cc == "Prv-CCL19"
m = prv & np.isin(pid_cc, use_cc)
print("CCL19, inside Prv-CCL19: technical-only pooled Spearman %.3f | XGBoost with genes %.3f" %
      (spearmanr(y_cc[m], tech_cc[m])[0], spearmanr(y_cc[m], pred_cc["xgboost"][m])[0]))

### 5c. CCL19 without CCL21 as an input

In [ ]:
j21 = genes.index("CCL21")
keep_cc2 = np.array([i for i in keep_cc if i != j21])
res_cc2, pred_cc2 = lopo(X_mur, y_cc, pid_cc, sub_cc, keep_cc2, use_cc, FAST_MODELS, "CCL19_without_CCL21")
print(summarize(res_cc2))                                                 # compare with section 4
res_cc2.to_csv(os.path.join(OUT, "ccl19_without_ccl21_per_patient.csv"), index=False)

### 5d. Nested tuning of Elastic Net

In [ ]:
if RUN_TUNING:
    res_t_lv, chosen_lv = tune_en(X_mac, y_lv, pid_lv, keep_lv, use_lv, "LYVE1_macrophages")
    res_t_cc, chosen_cc = tune_en(X_mur, y_cc, pid_cc, keep_cc, use_cc, "CCL19_mural")
    print("LYVE1 tuned:", res_t_lv[["spearman", "r2", "rmse", "mae"]].mean().round(3).to_dict())
    print("CCL19 tuned:", res_t_cc[["spearman", "r2", "rmse", "mae"]].mean().round(3).to_dict())
    pd.concat([res_t_lv, res_t_cc]).to_csv(os.path.join(OUT, "elastic_net_tuned_per_patient.csv"), index=False)
    print(pd.concat([chosen_lv.assign(target="LYVE1"), chosen_cc.assign(target="CCL19")]).to_string(index=False))

### 5e. Zero inflation: expressing vs zero (AUC) and ranking among expressing cells

In [ ]:
for name, res in [("LYVE1", res_lv), ("CCL19", res_cc)]:
    print(name)
    print(res[res["model"].isin(["elastic_net", "xgboost"])]
          .groupby("model")[["auc", "sp_pos"]].mean().round(3))

## 6. Genes that drive the Elastic Net predictions

In [ ]:
st_lv, n_lv = stable_genes(X_mac, y_lv, pid_lv, keep_lv, genes, use_lv)
st_cc, n_cc = stable_genes(X_mur, y_cc, pid_cc, keep_cc, genes, use_cc)
print("LYVE1: genes used in >=1 fold:", n_lv, "| in all folds:", len(st_lv))
print(st_lv.head(15).round(3)); print(st_lv.tail(8).round(3))
print("CCL19: genes used in >=1 fold:", n_cc, "| in all folds:", len(st_cc))
print(st_cc.head(15).round(3)); print(st_cc.tail(8).round(3))
st_lv.to_csv(os.path.join(OUT, "EN_stable_genes_LYVE1.csv"))
st_cc.to_csv(os.path.join(OUT, "EN_stable_genes_CCL19.csv"))
# A large coefficient means association with the target, not causation; folds overlap heavily,
# so stability across folds is only weak evidence.

## 7. External validation (LYVE1, independent cohort GSE213216)
Models are trained on all GSE179640 macrophages with fixed settings and applied to the external cells
without any tuning on external data.

In [ ]:
y_ex = X_ext[:, j_lv].toarray().ravel()
pid_ex = meta_ext["patient_id"].values
Xt_all = X_mac[:, keep_lv]
sel = top_var_genes(Xt_all)
A = Xt_all[:, sel].toarray(); mu, sd = A.mean(0), A.std(0) + 1e-8; A = (A - mu) / sd
B = (X_ext[:, keep_lv][:, sel].toarray() - mu) / sd

pred_ex = {}
for n in ["elastic_net", "xgboost"]:
    pred_ex[n] = MODELS[n]().fit(A, y_lv).predict(B)

cols = ["nCount_RNA", "nFeature_RNA", "percent.mt"]
Tf = lambda d: np.column_stack([np.log1p(d[cols[0]]), np.log1p(d[cols[1]]), d[cols[2]]])
Ttr, Tex = Tf(meta_mac), Tf(meta_ext)
t_mu, t_sd = Ttr.mean(0), Ttr.std(0) + 1e-8
pred_ex["tech_only"] = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                                    tree_method="hist", random_state=SEED
                                    ).fit((Ttr - t_mu) / t_sd, y_lv).predict((Tex - t_mu) / t_sd)
pred_ex["train_mean"] = np.full(len(y_ex), y_lv.mean())

use_ex = usable_patients(y_ex, pid_ex)
print(len(use_ex), "external patients:", use_ex)                        # expected 5
rows = []
for p in use_ex:
    mk = pid_ex == p
    for n, v in pred_ex.items():
        rows.append(dict(patient=p, model=n, n=int(mk.sum()), n_pos=int((y_ex[mk] > 0).sum()),
                         **metrics(y_ex[mk], v[mk])))
res_ex = pd.DataFrame(rows)
print(res_ex.pivot(index="patient", columns="model", values="spearman").round(3))
print(res_ex.groupby("model")[["spearman", "r2", "auc", "sp_pos"]].mean().round(3))
res_ex.to_csv(os.path.join(OUT, "lyve1_external_per_patient.csv"), index=False)
out_ex = meta_ext[["barcode", "patient_id", "tissue"]].copy(); out_ex["LYVE1"] = y_ex
for n, v in pred_ex.items():
    out_ex["pred_" + n] = v
out_ex.to_csv(os.path.join(OUT, "preds_external_LYVE1.csv"), index=False)

## 8. Figures

In [ ]:
names = {"baseline": "Baseline (subtype mean)", "elastic_net": "Elastic Net",
         "random_forest": "Random Forest", "xgboost": "XGBoost"}
colors = {"baseline": "#9e9e9e", "elastic_net": "#1f77b4", "random_forest": "#2ca02c", "xgboost": "#d62728"}

# --- Figure 1: per-patient Spearman, internal ---
fig1, axes = plt.subplots(1, 2, figsize=(15, 4.8), facecolor="white")
for ax, (title, res) in zip(axes, [("LYVE1 (macrophages)", res_lv), ("CCL19 (mural cells)", res_cc)]):
    tbl = res.pivot(index="patient", columns="model", values="spearman")
    order = [m_ for m_ in names if m_ in tbl.columns]
    w = 0.8 / len(order)
    for k, m_ in enumerate(order):
        ax.bar(np.arange(len(tbl)) + (k - (len(order) - 1) / 2) * w, tbl[m_].fillna(0), w,
               label=names[m_], color=colors[m_])
    ax.set_xticks(range(len(tbl))); ax.set_xticklabels(tbl.index)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(title); ax.set_ylabel("Spearman (test patient)"); ax.set_xlabel("Held-out patient")
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig1.savefig(os.path.join(OUT, "fig1_spearman_per_patient.png"), dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

# --- Figure 2: observed vs predicted by decile, internal (pooled test patients) ---
fig2, axes = plt.subplots(1, 2, figsize=(10, 4.2), facecolor="white")
for ax, (title, y_, pred_, use_, pid_) in zip(axes, [
        ("LYVE1 (macrophages)", y_lv, pred_lv, use_lv, pid_lv),
        ("CCL19 (mural cells)", y_cc, pred_cc, use_cc, pid_cc)]):
    mk = np.isin(pid_, use_); lim = 0
    for mdl in ["elastic_net", "xgboost"]:
        pr = pred_[mdl][mk]
        q = pd.qcut(pd.Series(pr).rank(method="first"), 10, labels=False)
        g = pd.DataFrame({"q": q.values, "pred": pr, "obs": y_[mk]}).groupby("q").mean()
        ax.plot(g["pred"], g["obs"], marker="o", color=colors[mdl], label=names[mdl])
        lim = max(lim, g["pred"].max(), g["obs"].max())
    ax.plot([0, lim], [0, lim], "k--", lw=1)
    ax.set_title(title); ax.set_xlabel("Mean predicted (decile)"); ax.set_ylabel("Mean observed")
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig2.savefig(os.path.join(OUT, "fig2_calibration_deciles.png"), dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

# --- Figure 3: external validation ---
fig3, ax = plt.subplots(1, 2, figsize=(13, 4.8), facecolor="white")
tabE = res_ex.pivot(index="patient", columns="model", values="spearman")
series = [("elastic_net", "Elastic Net", "#1f77b4"), ("xgboost", "XGBoost", "#d62728"),
          ("tech_only", "Technical metrics only (baseline)", "#9e9e9e")]
nE = pd.Series(pid_ex).value_counts()
w = 0.26
for k, (col, lab, colr) in enumerate(series):
    ax[0].bar(np.arange(len(use_ex)) + (k - 1) * w, tabE.loc[use_ex, col], w, label=lab, color=colr)
ax[0].set_xticks(range(len(use_ex)))
ax[0].set_xticklabels([f"{p}\n(n={nE[p]})" for p in use_ex])
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_ylabel("Spearman (test patient)"); ax[0].set_xlabel("External patient (GSE213216)")
ax[0].set_title("LYVE1 - external validation"); ax[0].legend(fontsize=8, frameon=False)
mk = np.isin(pid_ex, use_ex); lim = 0
for col, lab, colr in series[:2]:
    pr = pred_ex[col][mk]
    q = pd.qcut(pd.Series(pr).rank(method="first"), 10, labels=False)
    g = pd.DataFrame({"q": q.values, "pred": pr, "obs": y_ex[mk]}).groupby("q").mean()
    ax[1].plot(g["pred"], g["obs"], marker="o", color=colr, label=lab)
    lim = max(lim, g["pred"].max(), g["obs"].max())
ax[1].plot([0, lim], [0, lim], "k--", lw=1)
ax[1].set_xlabel("Mean predicted (decile)"); ax[1].set_ylabel("Mean observed")
ax[1].set_title(f"LYVE1 - external, pooled ({len(use_ex)} patients)"); ax[1].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig3.savefig(os.path.join(OUT, "fig3_external_validation.png"), dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

# --- Figure 4: summary (per-patient Spearman and where each gene can be modelled) ---
sp = lambda res, m_: res[res["model"] == m_].set_index("patient")["spearman"]
groups = [(f"LYVE1\ninternal (n={len(use_lv)})", sp(res_lv, "elastic_net"), sp(res_lv, "baseline")),
          (f"LYVE1\nexternal (n={len(use_ex)})", sp(res_ex, "elastic_net"), sp(res_ex, "tech_only")),
          (f"CCL19\ninternal (n={len(use_cc)})", sp(res_cc, "elastic_net"), sp(res_cc, "baseline"))]
base_names = ["subtype mean", "technical only", "subtype mean"]
# patients with >= 50 cells and >= 10 expressing cells per tissue, from the R target screening
# (results/03_target_screening/target_screen_summary_by_tissue.csv; All macrophages for LYVE1, All mural for CCL19)
tissues = ["Ctrl", "EuE", "EcP", "EcPA", "EcO"]
n_lv_t = [0, 4, 8, 3, 4]
n_cc_t = [1, 0, 3, 5, 0]

fig4, ax = plt.subplots(1, 2, figsize=(13, 5.8), facecolor="white")
rng = np.random.default_rng(0)
for i, (lab, en, base) in enumerate(groups):
    ax[0].scatter(i + rng.uniform(-0.12, 0.12, len(en)), en, color="#1f77b4", s=45, zorder=3)
    ax[0].hlines(en.mean(), i - 0.25, i + 0.25, color="#1f77b4", lw=2.5)
    ax[0].scatter([i + 0.18], [base.mean()], marker="D", color="#9e9e9e", s=60, zorder=3)
ax[0].scatter([], [], color="#1f77b4", s=45, label="Elastic Net (dot = one test patient, line = mean)")
ax[0].scatter([], [], marker="D", color="#9e9e9e", s=60, label="Mean of the baseline (type differs by group)")
ax[0].set_xticks(range(3))
ax[0].set_xticklabels([f"{g[0]}\nbaseline: {b}" for g, b in zip(groups, base_names)])
ax[0].axhline(0, color="k", lw=0.8); ax[0].set_ylim(-0.1, 1.0)
ax[0].set_ylabel("Spearman per test patient"); ax[0].set_title("Prediction per patient")
ax[0].legend(fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.30))
x = np.arange(len(tissues))
ax[1].bar(x - 0.2, n_lv_t, 0.4, label="LYVE1 (macrophages)", color="#1f77b4")
ax[1].bar(x + 0.2, n_cc_t, 0.4, label="CCL19 (mural cells)", color="#ff7f0e")
ax[1].set_xticks(x); ax[1].set_xticklabels(tissues)
ax[1].set_ylabel("Patients with enough expressing cells"); ax[1].set_title("Where each gene can be modelled")
ax[1].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig4.savefig(os.path.join(OUT, "fig4_LYVE1_vs_CCL19_summary.png"), dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## 9. Summary table and download

In [ ]:
summary = pd.concat([res_lv, res_cc, res_om, res_cc2, res_ex.assign(analysis="LYVE1_external")])
summary = (summary.groupby(["analysis", "model"])[["spearman", "r2", "rmse", "mae", "auc", "sp_pos"]]
           .mean().round(3).reset_index())
summary.to_csv(os.path.join(OUT, "summary_all_analyses.csv"), index=False)
print(summary.to_string(index=False))

zip_path = os.path.join(WORK, "ml_results.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for f in sorted(glob.glob(os.path.join(OUT, "*"))):
        z.write(f, os.path.basename(f))
print("files:", sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("zip saved at", zip_path)